# Manual Review Queue
Shows every record where `needs_manual_check = True` and highlights exactly which flags triggered it.

**Flag reference:**

| Flag | Meaning |
|---|---|
| `flag_math_total_used` | Valid + Invalid + No Vote ≠ Total Used |
| `flag_math_valid_score` | Sum of candidate scores ≠ Total Valid Ballots |
| `flag_name_mismatch` | Candidate name not found in master list |
| `flag_missing_data` | Score/ballot field could not be parsed (NaN) |
| `flag_linguistic_mismatch` | OCR score has conflicting digit vs Thai word |
| `flag_ocr_timeout` | OCR API timed out after all retries for a page/chunk |
| `flag_missing_counterpart` | Station missing its paired form type (set during aggregation, not in `needs_manual_check`) |

In [ ]:
import pandas as pd
from pathlib import Path

# Adjust path if running from a different directory
CSV_PATH = Path('../../output_data/master_summary_log.csv')

df = pd.read_csv(CSV_PATH)

# All flag columns present in the schema
FLAG_COLS = [
    'flag_math_total_used',
    'flag_math_valid_score',
    'flag_name_mismatch',
    'flag_missing_data',
    'flag_linguistic_mismatch',
    'flag_ocr_timeout',
    'flag_missing_counterpart',
]

# Coerce all flag columns to bool (CSV stores 'True'/'False' strings or NaN)
for col in FLAG_COLS:
    if col in df.columns:
        df[col] = df[col].map(lambda x: str(x).strip().lower() == 'true')
    else:
        df[col] = False

if 'needs_manual_check' in df.columns:
    df['needs_manual_check'] = df['needs_manual_check'].map(lambda x: str(x).strip().lower() == 'true')

print(f'Total records : {len(df)}')
print(f'Need manual check : {df["needs_manual_check"].sum()}')
print(f'Columns: {list(df.columns)}')

## Summary — how many records triggered each flag

In [ ]:
summary = pd.DataFrame({
    'flag': FLAG_COLS,
    'count': [df[c].sum() for c in FLAG_COLS],
    'pct_of_all': [f"{df[c].mean()*100:.1f}%" for c in FLAG_COLS],
})
summary

## Manual Review Queue
One row per record that needs human verification.  
Red = flag raised &nbsp;|&nbsp; Green = flag clear

In [ ]:
IDENTITY_COLS = ['amphoe', 'tambon', 'unit', 'type']

# Records where needs_manual_check is True
queue = (
    df[df['needs_manual_check']]
    .copy()
    .sort_values(['tambon', 'unit', 'type'])
    .reset_index(drop=True)
)

# Show only flags that cover needs_manual_check (exclude flag_missing_counterpart here)
MANUAL_FLAGS = [
    'flag_math_total_used',
    'flag_math_valid_score',
    'flag_name_mismatch',
    'flag_missing_data',
    'flag_linguistic_mismatch',
    'flag_ocr_timeout',
]

display_cols = IDENTITY_COLS + MANUAL_FLAGS + ['details']
display_cols = [c for c in display_cols if c in queue.columns]

def style_flag(val):
    if val is True:
        return 'background-color: #e74c3c; color: white; font-weight: bold'
    if val is False:
        return 'background-color: #a9dfbf; color: #1a1a1a'
    return ''

flag_cols_present = [c for c in MANUAL_FLAGS if c in queue.columns]

print(f'{len(queue)} record(s) in the manual review queue\n')

(
    queue[display_cols]
    .style
    .map(style_flag, subset=flag_cols_present)
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-size', '11px'), ('text-align', 'left'),
                  ('background-color', '#2c3e50'), ('color', 'white')]
    }])
)

## Which flags are active per record (flag fingerprint)

In [ ]:
# Build a human-readable list of raised flags for each record
SHORT = {
    'flag_math_total_used':     'math_total',
    'flag_math_valid_score':    'math_score',
    'flag_name_mismatch':       'name_mismatch',
    'flag_missing_data':        'missing_data',
    'flag_linguistic_mismatch': 'linguistic',
    'flag_ocr_timeout':         'ocr_timeout',
}

def active_flags(row):
    return ', '.join(SHORT[c] for c in MANUAL_FLAGS if c in row and row[c])

queue['active_flags'] = queue.apply(active_flags, axis=1)
queue['flag_count']   = queue[[c for c in MANUAL_FLAGS if c in queue.columns]].sum(axis=1)

fingerprint_cols = IDENTITY_COLS + ['flag_count', 'active_flags', 'details']
fingerprint_cols = [c for c in fingerprint_cols if c in queue.columns]

queue[fingerprint_cols].sort_values('flag_count', ascending=False).reset_index(drop=True)

In [12]:
queue[MANUAL_FLAGS].sum(axis=0)

flag_math_total_used         97
flag_math_valid_score       219
flag_name_mismatch          241
flag_missing_data           234
flag_linguistic_mismatch      0
flag_ocr_timeout              0
dtype: int64

## Stations also missing their counterpart form

In [ ]:
# Records in the manual queue that ALSO have flag_missing_counterpart
both = queue[queue['flag_missing_counterpart'] == True] if 'flag_missing_counterpart' in queue.columns else pd.DataFrame()

if len(both):
    print(f'{len(both)} record(s) need manual check AND are missing their counterpart form:\n')
    print(both[IDENTITY_COLS + ['active_flags']].to_string(index=False))
else:
    print('No records in the manual queue are also missing a counterpart form.')

# Separately: records flagged ONLY for missing counterpart (no other flag)
counterpart_only = df[
    (df.get('flag_missing_counterpart', False) == True) & (~df['needs_manual_check'])
]
print(f'\n{len(counterpart_only)} station(s) flagged only for missing counterpart (not in manual queue):')
if len(counterpart_only):
    print(counterpart_only[IDENTITY_COLS].to_string(index=False))

## Export manual queue to CSV

In [ ]:
out_path = Path('../../output_data/manual_review_queue.csv')
queue[fingerprint_cols].sort_values('flag_count', ascending=False).to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved {len(queue)} records → {out_path}')